In [ ]:
print("Starting notebook")

In [ ]:
print("Starting RSNA Knee Axial Supervised Baseline")

"""
RSNA Knee Abnormality Detection
--------------------------------

Supervised axial-only baseline using weak labels extracted from
the LLM/radiology-report label dataset.

IMPORTANT:
- We intentionally use Chunk 1 for training and Chunk 2 for testing.
- We do NOT care about patient/study leakage for this experiment.
- Existing tensor caches are reused.
- Only one image tensor chunk is kept in RAM at a time.
- GPU memory is protected with AMP + moderate batch size.
- Labels are study-level and therefore repeated for every axial slice.
- Study-level prediction is obtained by top-k slice pooling.

Targets:
    ACL
    MCL
    Medial Meniscus
    Lateral Meniscus
    Medial OA
    Lateral OA
    PF OA
    Effusion
    Synovitis
    Baker's
    Contusion
    Fracture

LLM labels:
    probabilities are converted to binary using > 0.5

Outputs:
    /kaggle/working/knee_classifier_weights.pt
    /kaggle/working/knee_classifier_results.json
"""

import os
import glob
import json
import time
import gc
import re
import warnings

from typing import Dict, List, Tuple, Optional

import cv2
import numpy as np
import pandas as pd
import pydicom

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore")


# =====================================================================
# 1. CONFIGURATION
# =====================================================================

COMPETITION_PATH = (
    "/kaggle/input/competitions/rsna-knee-abnormality-detection"
)

LLM_DIR = (
    "/kaggle/input/datasets/stevenleehans/rsna-knee-llm-report-labels"
)

AXIAL_PATHS_CACHE = (
    "/kaggle/working/axial_paths_cache.json"
)

CHUNK_FILE_PATTERN = (
    "/kaggle/working/axial_chunk_{}.pt"
)

LABEL_CHUNK_FILE_PATTERN = (
    "/kaggle/working/axial_labels_chunk_{}.pt"
)

MODEL_WEIGHTS_FILE = (
    "/kaggle/working/knee_classifier_weights.pt"
)

RESULTS_FILE = (
    "/kaggle/working/knee_classifier_results.json"
)

LABEL_MAPPING_FILE = (
    "/kaggle/working/knee_label_mapping.json"
)

TARGET_RESOLUTION = (128, 128)

TARGET_COLUMNS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

# ---------------------------------------------------------------------
# Training configuration
# ---------------------------------------------------------------------

EPOCHS = 8

BATCH_SIZE = 128

LEARNING_RATE = 3e-4

WEIGHT_DECAY = 1e-4

NUM_WORKERS = 2

PIN_MEMORY = True

# Number of slices used for study-level pooling.
# top-k=5 means the 5 strongest abnormal-looking slices are averaged.
TOP_K = 5

# AMP significantly reduces T4 memory usage.
USE_AMP = True

SEED = 42


# =====================================================================
# 2. REPRODUCIBILITY
# =====================================================================

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("CONFIGURATION")
print("=" * 70)
print(f"Device       : {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(
        f"GPU memory   : "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

print(f"Resolution   : {TARGET_RESOLUTION}")
print(f"Batch size   : {BATCH_SIZE}")
print(f"Epochs       : {EPOCHS}")
print(f"AMP          : {USE_AMP}")
print(f"Top-k pooling: {TOP_K}")
print("=" * 70)


# =====================================================================
# 3. FIND LLM LABEL FILE
# =====================================================================

def find_label_files() -> List[str]:
    """
    Find CSV/Parquet/JSON files in the supplied LLM label directory.
    """

    patterns = [
        "*.csv",
        "*.CSV",
        "*.parquet",
        "*.json",
        "*.jsonl",
    ]

    files = []

    for pattern in patterns:
        files.extend(
            glob.glob(
                os.path.join(LLM_DIR, "**", pattern),
                recursive=True,
            )
        )

    return sorted(set(files))


def inspect_label_files():
    """
    Print available files and their columns.

    This is intentionally done before guessing the schema.
    """

    files = find_label_files()

    print("\n" + "=" * 70)
    print("LLM LABEL DATASET INSPECTION")
    print("=" * 70)

    if not files:
        raise FileNotFoundError(
            f"No CSV/Parquet/JSON files found in:\n{LLM_DIR}"
        )

    for file_path in files:
        print(f"\nFILE: {file_path}")

        try:
            if file_path.endswith(".csv"):
                df = pd.read_csv(file_path, nrows=5)

            elif file_path.endswith(".parquet"):
                df = pd.read_parquet(file_path).head()

            elif file_path.endswith(".json"):
                df = pd.read_json(file_path).head()

            elif file_path.endswith(".jsonl"):
                df = pd.read_json(
                    file_path,
                    lines=True,
                    nrows=5,
                )

            else:
                continue

            print("Columns:")
            print(list(df.columns))

        except Exception as e:
            print(f"Could not inspect: {e}")

    print("=" * 70)

    return files


# =====================================================================
# 4. NORMALIZE COLUMN NAMES
# =====================================================================

def normalize_column_name(x: str) -> str:
    """
    Convert column names into a comparable representation.
    """

    x = str(x).lower().strip()

    x = x.replace("_", " ")
    x = x.replace("-", " ")
    x = re.sub(r"\s+", " ", x)

    return x


def find_column(
    columns,
    candidates: List[str],
) -> Optional[str]:
    """
    Find a column using normalized exact matching first,
    then substring matching.
    """

    normalized = {
        normalize_column_name(c): c
        for c in columns
    }

    # Exact match
    for candidate in candidates:

        candidate_norm = normalize_column_name(candidate)

        if candidate_norm in normalized:
            return normalized[candidate_norm]

    # Substring match
    for candidate in candidates:

        candidate_norm = normalize_column_name(candidate)

        for norm_col, original_col in normalized.items():

            if candidate_norm in norm_col:
                return original_col

    return None


# =====================================================================
# 5. LOAD LLM LABELS
# =====================================================================

def load_label_dataframe() -> pd.DataFrame:
    """
    Load the most likely label table.

    We prefer a table containing:
        - StudyInstanceUID
        - multiple target columns
    """

    files = find_label_files()

    candidates = []

    for file_path in files:

        try:

            if file_path.endswith(".csv"):
                df = pd.read_csv(file_path)

            elif file_path.endswith(".parquet"):
                df = pd.read_parquet(file_path)

            elif file_path.endswith(".json"):
                df = pd.read_json(file_path)

            elif file_path.endswith(".jsonl"):
                df = pd.read_json(
                    file_path,
                    lines=True,
                )

            else:
                continue

            columns = list(df.columns)

            uid_col = find_column(
                columns,
                [
                    "StudyInstanceUID",
                    "Study Instance UID",
                    "study_uid",
                    "studyinstanceuid",
                    "study id",
                ],
            )

            target_matches = 0

            for target in TARGET_COLUMNS:

                aliases = [
                    target,
                    target.lower(),
                    target.replace(" ", "_"),
                ]

                if target == "Baker's":
                    aliases += [
                        "Baker cyst",
                        "Bakers cyst",
                        "Baker's cyst",
                        "Baker",
                    ]

                if target == "Medial Meniscus":
                    aliases += [
                        "medial meniscus tear",
                        "medial meniscus injury",
                    ]

                if target == "Lateral Meniscus":
                    aliases += [
                        "lateral meniscus tear",
                        "lateral meniscus injury",
                    ]

                if target == "Medial OA":
                    aliases += [
                        "medial osteoarthritis",
                        "medial oa",
                    ]

                if target == "Lateral OA":
                    aliases += [
                        "lateral osteoarthritis",
                        "lateral oa",
                    ]

                if target == "PF OA":
                    aliases += [
                        "patellofemoral oa",
                        "patellofemoral osteoarthritis",
                        "pf osteoarthritis",
                    ]

                if find_column(columns, aliases):
                    target_matches += 1

            if uid_col is not None and target_matches >= 2:

                candidates.append(
                    (
                        target_matches,
                        len(df),
                        file_path,
                        df,
                        uid_col,
                    )
                )

        except Exception as e:

            print(
                f"Could not read {file_path}: {e}"
            )

    if not candidates:

        raise RuntimeError(
            "\nCould not automatically identify the LLM label table.\n"
            "Run the notebook once and inspect the printed columns above."
        )

    candidates.sort(
        key=lambda x: (x[0], x[1]),
        reverse=True,
    )

    best = candidates[0]

    print("\n" + "=" * 70)
    print("SELECTED LLM LABEL FILE")
    print("=" * 70)

    print(f"File        : {best[2]}")
    print(f"Rows        : {len(best[3]):,}")
    print(f"UID column  : {best[4]}")
    print(f"Target cols : {best[0]}")
    print("=" * 70)

    return best[3]


def build_label_lookup(
    df: pd.DataFrame,
) -> Tuple[Dict[str, np.ndarray], Dict[str, str]]:
    """
    Build:

        StudyInstanceUID -> 12-element binary vector

    Values > 0.5 become 1.

    Missing labels become 0.
    """

    columns = list(df.columns)

    uid_col = find_column(
        columns,
        [
            "StudyInstanceUID",
            "Study Instance UID",
            "study_uid",
            "studyinstanceuid",
            "study id",
        ],
    )

    if uid_col is None:
        raise RuntimeError(
            "Could not find StudyInstanceUID in label dataframe."
        )

    selected_columns = {}

    for target in TARGET_COLUMNS:

        aliases = [
            target,
            target.lower(),
            target.replace(" ", "_"),
        ]

        if target == "Baker's":
            aliases += [
                "Baker cyst",
                "Bakers cyst",
                "Baker's cyst",
                "Baker",
            ]

        if target == "Medial Meniscus":
            aliases += [
                "medial meniscus tear",
                "medial meniscus injury",
            ]

        if target == "Lateral Meniscus":
            aliases += [
                "lateral meniscus tear",
                "lateral meniscus injury",
            ]

        if target == "Medial OA":
            aliases += [
                "medial osteoarthritis",
            ]

        if target == "Lateral OA":
            aliases += [
                "lateral osteoarthritis",
            ]

        if target == "PF OA":
            aliases += [
                "patellofemoral oa",
                "patellofemoral osteoarthritis",
            ]

        col = find_column(columns, aliases)

        selected_columns[target] = col

    print("\nLABEL COLUMN MAPPING")
    print("-" * 70)

    for target, col in selected_columns.items():
        print(
            f"{target:20s} -> "
            f"{str(col)}"
        )

    print("-" * 70)

    # -------------------------------------------------------------
    # Build lookup
    # -------------------------------------------------------------

    lookup = {}

    for _, row in df.iterrows():

        uid = str(row[uid_col])

        if uid == "nan":
            continue

        labels = []

        for target in TARGET_COLUMNS:

            col = selected_columns[target]

            if col is None:

                # Missing target column.
                # We use 0, but report it clearly.
                value = 0.0

            else:

                value = pd.to_numeric(
                    row[col],
                    errors="coerce",
                )

                if pd.isna(value):
                    value = 0.0

            # Requested threshold:
            # probability > .5 -> 1
            label = 1.0 if float(value) > 0.5 else 0.0

            labels.append(label)

        lookup[uid] = np.asarray(
            labels,
            dtype=np.float32,
        )

    print(
        f"Built label lookup for "
        f"{len(lookup):,} studies."
    )

    # Save mapping for reproducibility
    with open(
        LABEL_MAPPING_FILE,
        "w",
    ) as f:

        json.dump(
            {
                "uid_column": uid_col,
                "columns": selected_columns,
                "threshold": 0.5,
                "targets": TARGET_COLUMNS,
            },
            f,
            indent=2,
        )

    return lookup, selected_columns


# =====================================================================
# 6. AXIAL PATH EXTRACTION
# =====================================================================

def get_axial_paths_from_csv() -> List[str]:
    """
    Load cached axial paths if available.

    Otherwise construct them from train_series.csv.
    """

    if os.path.exists(AXIAL_PATHS_CACHE):

        print(
            f"Loading cached paths:\n"
            f"{AXIAL_PATHS_CACHE}"
        )

        with open(
            AXIAL_PATHS_CACHE,
            "r",
        ) as f:

            return json.load(f)

    print(
        "Axial path cache does not exist."
    )

    train_series_csv = (
        f"{COMPETITION_PATH}/train_series.csv"
    )

    df_series = pd.read_csv(
        train_series_csv
    )

    axial_df = df_series[
        df_series["Anatomical_Plane"]
        .astype(str)
        .str.lower()
        .eq("axial")
    ]

    print(
        f"Total series : {len(df_series):,}"
    )

    print(
        f"Axial series : {len(axial_df):,}"
    )

    axial_paths = []

    for _, row in axial_df.iterrows():

        study_uid = row["StudyInstanceUID"]
        series_uid = row["SeriesInstanceUID"]

        series_dir = (
            f"{COMPETITION_PATH}/train_series/"
            f"{study_uid}/{series_uid}"
        )

        axial_paths.extend(
            glob.glob(
                f"{series_dir}/*.dcm"
            )
        )

    with open(
        AXIAL_PATHS_CACHE,
        "w",
    ) as f:

        json.dump(
            axial_paths,
            f,
        )

    print(
        f"Found {len(axial_paths):,} axial DICOMs."
    )

    return axial_paths


# =====================================================================
# 7. STUDY UID FROM PATH
# =====================================================================

def study_uid_from_path(path: str) -> str:
    """
    Expected:

    .../train_series/<StudyUID>/<SeriesUID>/<file>.dcm

    Therefore StudyUID is parent-parent directory.
    """

    return os.path.basename(
        os.path.dirname(
            os.path.dirname(path)
        )
    )


# =====================================================================
# 8. CREATE LABEL TENSOR WITHOUT READING DICOM
# =====================================================================

def build_chunk_labels(
    paths: List[str],
    chunk_id: int,
    label_lookup: Dict[str, np.ndarray],
) -> torch.Tensor:
    """
    Generate labels from paths.

    This does NOT read the DICOM files.
    """

    label_cache = (
        LABEL_CHUNK_FILE_PATTERN.format(
            chunk_id
        )
    )

    if os.path.exists(label_cache):

        print(
            f"Loading cached labels:"
            f" {label_cache}"
        )

        return torch.load(
            label_cache,
            map_location="cpu",
        )

    print(
        f"Building labels for Chunk "
        f"{chunk_id}..."
    )

    labels = np.zeros(
        (
            len(paths),
            len(TARGET_COLUMNS),
        ),
        dtype=np.float32,
    )

    missing = 0

    for i, path in enumerate(paths):

        uid = study_uid_from_path(path)

        if uid in label_lookup:

            labels[i] = label_lookup[uid]

        else:

            missing += 1

    tensor = torch.from_numpy(labels)

    torch.save(
        tensor,
        label_cache,
    )

    print(
        f"Saved labels: {label_cache}"
    )

    print(
        f"Missing study labels: "
        f"{missing:,}/{len(paths):,}"
    )

    return tensor


# =====================================================================
# 9. LOAD IMAGE CHUNK
# =====================================================================

def load_chunk_tensor(
    paths: List[str],
    chunk_id: int,
) -> torch.Tensor:
    """
    Load an image tensor from disk if cached.

    Otherwise read DICOMs once and cache the resulting tensor.

    WARNING:
        A 58k x 128 x 128 float32 tensor is approximately 3.8 GB.
        This is intentional, but only ONE chunk should be resident.
    """

    cache_pt = (
        CHUNK_FILE_PATTERN.format(
            chunk_id
        )
    )

    if os.path.exists(cache_pt):

        print(
            f"\nLoading existing image tensor:"
            f"\n{cache_pt}"
        )

        tensor = torch.load(
            cache_pt,
            map_location="cpu",
        )

        print(
            f"Chunk {chunk_id} shape: "
            f"{tuple(tensor.shape)}"
        )

        return tensor

    print(
        f"\nReading DICOMs for Chunk "
        f"{chunk_id}: {len(paths):,}"
    )

    start = time.time()

    slices = []

    for idx, file_path in enumerate(paths):

        if idx % 1000 == 0:

            elapsed = time.time() - start

            rate = (
                idx / elapsed
                if elapsed > 0
                else 0
            )

            remaining = (
                len(paths) - idx
            )

            eta = (
                remaining / rate
                if rate > 0
                else 0
            )

            print(
                f"Chunk {chunk_id}: "
                f"{idx:,}/{len(paths):,} | "
                f"ETA {eta/60:.1f} min"
            )

        try:

            ds = pydicom.dcmread(
                file_path,
                force=True,
            )

            img = ds.pixel_array.astype(
                np.float32
            )

            # Per-slice intensity normalization
            mn = img.min()
            mx = img.max()

            if mx > mn:

                img = (
                    img - mn
                ) 